# Partial-Cycle Battery SoH Estimation via Truncation-Aware Transformer with Monotonicity Constraint
**Dataset:** Severson et al. (2019) — MATR/Stanford 124-cell LFP dataset  
*Severson, K.A. et al. "Data-driven prediction of battery cycle life before capacity fade." Nature Energy, 2019.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 150})
import os, warnings, h5py
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

## 1. Data Loading — Severson/MATR HDF5 Format

In [ ]:
# ── Locate the .mat / .h5 batch files ────────────────────────────────────────
# The MATR dataset ships as three batch files:
#   2017-05-12_batchdata_updated_struct_errorcorrect.mat   (batch 1, 41 cells)
#   2017-06-30_batchdata_updated_struct_errorcorrect.mat   (batch 2, 43 cells)
#   2018-04-12_batchdata_updated_struct_errorcorrect.mat   (batch 3, 40 cells)
#
# On Kaggle, upload all three as a single dataset and point DATA_DIR here.
# They are MATLAB v7.3 files, which h5py reads natively.

DATA_DIR = '/kaggle/input/severson-battery-dataset'

BATCH_FILES = [
    '2017-05-12_batchdata_updated_struct_errorcorrect.mat',
    '2017-06-30_batchdata_updated_struct_errorcorrect.mat',
    '2018-04-12_batchdata_updated_struct_errorcorrect.mat',
]

def h5str(ref, f):
    """Decode an HDF5 string reference or byte array to a Python string."""
    if isinstance(ref, h5py.Reference):
        return ''.join(chr(c) for c in f[ref][:].ravel())
    if isinstance(ref, (bytes, np.bytes_)):
        return ref.decode('utf-8')
    return str(ref)

def load_severson_batch(filepath):
    """
    Parse one MATR batch file into a list of per-cell dicts.
    Each dict has:
        cell_id   : str  (e.g. 'b1c0')
        cycles    : list of dicts with keys Qd, V, I, T, t
        Qd_first  : float  initial discharge capacity (Ah) — SoH denominator
    """
    cells = []
    with h5py.File(filepath, 'r') as f:
        batch = f['batch']
        n_cells = batch['summary'].shape[0] if 'summary' in batch else 0

        # Cell names are stored as object references in batch['cell_names']
        cell_names_refs = batch['cell_names'][:].ravel()

        for ci in range(len(cell_names_refs)):
            try:
                cid = h5str(cell_names_refs[ci], f)
            except Exception:
                cid = f'cell_{ci}'

            cyc_ref = batch['cycles'][ci, 0]
            cyc_grp = f[cyc_ref]

            cycle_list = []
            n_cycles = cyc_grp['Qd'].shape[0]

            for k in range(n_cycles):
                try:
                    Qd = f[cyc_grp['Qd'][k, 0]][:].ravel().astype(np.float32)
                    V  = f[cyc_grp['V' ][k, 0]][:].ravel().astype(np.float32)
                    I  = f[cyc_grp['I' ][k, 0]][:].ravel().astype(np.float32)
                    T  = f[cyc_grp['T' ][k, 0]][:].ravel().astype(np.float32)
                    t  = f[cyc_grp['t' ][k, 0]][:].ravel().astype(np.float32)
                except Exception:
                    continue

                if len(V) < 10:
                    continue
                cycle_list.append(dict(Qd=Qd, V=V, I=I, T=T, t=t,
                                       cap=float(Qd.max()) if len(Qd) else 0.0))

            if len(cycle_list) < 5:
                continue

            # Initial capacity = mean of first 5 cycles (more stable than cycle 0 alone)
            Qd_first = np.mean([c['cap'] for c in cycle_list[:5]])
            if Qd_first < 0.5:     # filter cells with abnormally low initial capacity
                continue

            cells.append(dict(cell_id=cid, cycles=cycle_list, Qd_first=Qd_first))

    return cells

print('Loading Severson batch files...')
all_cells = []
for bf in BATCH_FILES:
    fp = os.path.join(DATA_DIR, bf)
    if not os.path.isfile(fp):
        print(f'  [SKIP] not found: {bf}')
        continue
    batch_cells = load_severson_batch(fp)
    all_cells.extend(batch_cells)
    print(f'  Loaded {len(batch_cells)} cells from {bf}')

print(f'\nTotal cells: {len(all_cells)}')

In [ ]:
TRUNC_LEVELS = [0.30, 0.50, 0.75, 1.00]
N_FEATURES_PER_LEVEL = 8

def extract_features_from_cycle(cyc_dict, tau):
    """Extract 8 physics-motivated features from the first tau% of a cycle dict."""
    T   = len(cyc_dict['V'])
    n   = max(5, int(tau * T))
    V   = cyc_dict['V'][:n]
    I   = cyc_dict['I'][:n]
    Tmp = cyc_dict['T'][:n]
    t   = cyc_dict['t'][:n]
    dV  = np.diff(V) if len(V) > 1 else np.array([0.0], dtype=np.float32)

    f1 = float(np.mean(V))
    f2 = float(np.min(V))
    f3 = float(np.mean(I))
    f4 = float(np.mean(Tmp))
    f5 = float(np.trapz(np.abs(I), t)) if len(t) > 1 else 0.0   # charge throughput
    f6 = float(np.mean(dV))
    f7 = float(np.std(V))
    f8 = float((V[0] - V[-1]) / max(float(t[-1]), 1e-6))         # voltage drop rate
    return [f1, f2, f3, f4, f5, f6, f7, f8]

def build_feature_matrix(all_cells, trunc_levels):
    rows, sohs, bat_ids, cyc_nums = [], [], [], []

    for cell in all_cells:
        cid      = cell['cell_id']
        Qd_first = cell['Qd_first']
        n_cyc    = len(cell['cycles'])

        for k, cyc in enumerate(cell['cycles']):
            cap = cyc['cap']
            if cap <= 0:
                continue
            soh = cap / Qd_first

            feat_vec = []
            for tau in trunc_levels:
                feat_vec.extend(extract_features_from_cycle(cyc, tau))
            feat_vec.append((k + 1) / n_cyc)   # normalized cycle index

            rows.append(feat_vec)
            sohs.append(soh)
            bat_ids.append(cid)
            cyc_nums.append(k + 1)

    X = np.array(rows, dtype=np.float32)
    y = np.array(sohs, dtype=np.float32)
    return X, y, bat_ids, cyc_nums

print('Extracting multi-truncation features...')
X_all, y_all, bat_ids, cyc_nums = build_feature_matrix(all_cells, TRUNC_LEVELS)
print(f'Feature matrix : {X_all.shape}')
print(f'SoH range      : [{y_all.min():.3f}, {y_all.max():.3f}]')
print(f'Unique cells   : {len(set(bat_ids))}')

## 2. Multi-Truncation Feature Extraction

In [ ]:
TRUNC_LEVELS = [0.30, 0.50, 0.75, 1.00]
N_FEATURES_PER_LEVEL = 8

def extract_features(seg, ts):
    """Extract 8 physics-motivated features from a discharge segment."""
    V    = seg['Voltage_measured'].values
    I    = seg['Current_measured'].values
    Temp = seg['Temperature_measured'].values
    t    = seg['Time'].values
    dV   = np.diff(V) if len(V) > 1 else np.array([0.0])
    dt   = np.diff(t) if len(t) > 1 else np.array([1.0])

    f1 = np.mean(V)                                          # mean voltage
    f2 = np.min(V)                                           # min voltage
    f3 = np.mean(I)                                          # mean current
    f4 = np.mean(Temp)                                       # mean temperature
    f5 = np.trapz(np.abs(I), t) if len(t) > 1 else 0.0     # charge throughput
    f6 = np.mean(dV)                                         # mean dV
    f7 = np.std(V)                                           # voltage std
    f8 = (V[0] - V[-1]) / max(t[-1], 1e-6)                  # voltage drop rate
    return [f1, f2, f3, f4, f5, f6, f7, f8]

def build_feature_matrix(discharge_df, csv_dir, trunc_levels):
    rows, sohs, bat_ids, cyc_nums = [], [], [], []
    n_feat = len(trunc_levels) * N_FEATURES_PER_LEVEL + 1  # +1 for cycle_norm

    for _, row in discharge_df.iterrows():
        fpath = os.path.join(csv_dir, row['filename'])
        if not os.path.isfile(fpath):
            continue
        try:
            cyc = pd.read_csv(fpath)
        except Exception:
            continue

        needed = ['Voltage_measured','Current_measured','Temperature_measured',
                  'Voltage_load','Time']
        if not all(c in cyc.columns for c in needed):
            continue
        cyc = cyc[needed].ffill().bfill()
        T = len(cyc)
        if T < 10:
            continue

        feat_vec = []
        for tau in trunc_levels:
            n_pts = max(5, int(tau * T))
            seg   = cyc.iloc[:n_pts]
            feat_vec.extend(extract_features(seg, seg['Time'].values))
        feat_vec.append(row['cycle_norm'])  # cycle index feature

        rows.append(feat_vec)
        sohs.append(row['SoH'])
        bat_ids.append(row['battery_id'])
        cyc_nums.append(row['cycle_num'])

    X = np.array(rows, dtype=np.float32)
    y = np.array(sohs, dtype=np.float32)
    return X, y, bat_ids, cyc_nums

print('Extracting features...')
X_all, y_all, bat_ids, cyc_nums = build_feature_matrix(discharge, CSV_DIR, TRUNC_LEVELS)
print(f'Feature matrix: {X_all.shape} | SoH range: [{y_all.min():.3f}, {y_all.max():.3f}]')

## 3. Per-Battery Chronological Split (70/15/15)

In [ ]:
def per_battery_split(X, y, bat_ids, cyc_nums, train_frac=0.70, val_frac=0.15):
    bat_ids  = np.array(bat_ids)
    cyc_nums = np.array(cyc_nums)
    tr_idx, vl_idx, te_idx = [], [], []

    for bat in np.unique(bat_ids):
        mask = bat_ids == bat
        idx  = np.where(mask)[0]
        # sort by cycle number within battery
        order = np.argsort(cyc_nums[idx])
        idx   = idx[order]
        n     = len(idx)
        n_tr  = int(train_frac * n)
        n_vl  = int(val_frac * n)
        tr_idx.extend(idx[:n_tr])
        vl_idx.extend(idx[n_tr:n_tr+n_vl])
        te_idx.extend(idx[n_tr+n_vl:])

    return (X[tr_idx], y[tr_idx], bat_ids[tr_idx], cyc_nums[tr_idx],
            X[vl_idx], y[vl_idx],
            X[te_idx], y[te_idx], bat_ids[te_idx], cyc_nums[te_idx])

(X_tr, y_tr, bat_tr, cyc_tr,
 X_vl, y_vl,
 X_te, y_te, bat_te, cyc_te) = per_battery_split(X_all, y_all, bat_ids, cyc_nums)

# Normalize features only (y already in [0,1])
scaler = MinMaxScaler()
X_tr = scaler.fit_transform(X_tr).astype(np.float32)
X_vl = scaler.transform(X_vl).astype(np.float32)
X_te = scaler.transform(X_te).astype(np.float32)

print(f'Train: {len(y_tr)} | Val: {len(y_vl)} | Test: {len(y_te)}')

## 4. Metrics Helper

In [ ]:
def calc_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true).ravel(), np.array(y_pred).ravel()
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    return rmse, mae, mape

results = {}  # store all model results for final table

## 5. Baseline: Linear Regression

In [ ]:
lr = Ridge(alpha=1.0)
lr.fit(X_tr, y_tr)
y_pred_lr = lr.predict(X_te)
rmse_lr, mae_lr, mape_lr = calc_metrics(y_te, y_pred_lr)
results['Linear Regression'] = dict(rmse=rmse_lr, mae=mae_lr, mape=mape_lr, pred=y_pred_lr)
print(f'Linear Regression — RMSE: {rmse_lr:.4f} | MAE: {mae_lr:.4f} | MAPE: {mape_lr:.2f}%')

## 6. Baseline: MLP

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_model(model, X_tr, y_tr, X_vl, y_vl, epochs=200, lr=1e-3, batch=32):
    model = model.to(device)
    opt   = optim.Adam(model.parameters(), lr=lr)
    sched = optim.lr_scheduler.StepLR(opt, step_size=60, gamma=0.5)
    ds    = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
    dl    = DataLoader(ds, batch_size=batch, shuffle=True)
    Xv    = torch.from_numpy(X_vl).to(device)
    yv    = torch.from_numpy(y_vl).to(device)

    train_losses, val_losses = [], []
    for epoch in range(epochs):
        model.train()
        ep_loss = 0
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = nn.MSELoss()(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
        sched.step()
        train_losses.append(ep_loss / len(dl))
        model.eval()
        with torch.no_grad():
            val_losses.append(nn.MSELoss()(model(Xv), yv).item())
    return train_losses, val_losses

torch.manual_seed(42)
mlp = MLP(X_tr.shape[1])
train_model(mlp, X_tr, y_tr, X_vl, y_vl)
mlp.eval()
with torch.no_grad():
    y_pred_mlp = mlp(torch.from_numpy(X_te).to(device)).cpu().numpy()
rmse_mlp, mae_mlp, mape_mlp = calc_metrics(y_te, y_pred_mlp)
results['MLP'] = dict(rmse=rmse_mlp, mae=mae_mlp, mape=mape_mlp, pred=y_pred_mlp)
print(f'MLP — RMSE: {rmse_mlp:.4f} | MAE: {mae_mlp:.4f} | MAPE: {mape_mlp:.2f}%')

## 7. Baseline: LSTM

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        # 4 truncation levels → 4 timesteps of 8 features each
        self.lstm    = nn.LSTM(8, hidden, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        # +1: cycle_norm appended to final hidden state so LSTM gets same info as TAT/MLP
        self.fc      = nn.Sequential(nn.Linear(hidden + 1, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        x_seq   = x[:, :32].view(x.size(0), 4, 8)  # [B, 4, 8]
        x_cycle = x[:, 32:33]                        # [B, 1]
        out, _  = self.lstm(x_seq)
        h       = self.dropout(out[:, -1, :])        # [B, hidden]
        h       = torch.cat([h, x_cycle], dim=1)     # [B, hidden+1]
        return self.fc(h).squeeze(-1)

torch.manual_seed(42)
lstm_model = LSTMModel()
train_model(lstm_model, X_tr, y_tr, X_vl, y_vl)
lstm_model.eval()
with torch.no_grad():
    y_pred_lstm = lstm_model(torch.from_numpy(X_te).to(device)).cpu().numpy()
rmse_lstm, mae_lstm, mape_lstm = calc_metrics(y_te, y_pred_lstm)
results['LSTM'] = dict(rmse=rmse_lstm, mae=mae_lstm, mape=mape_lstm, pred=y_pred_lstm)
print(f'LSTM — RMSE: {rmse_lstm:.4f} | MAE: {mae_lstm:.4f} | MAPE: {mape_lstm:.2f}%')

## 8. Proposed: Truncation-Aware Transformer (TAT) with Monotonicity Loss

In [ ]:
class TruncationAwareTransformer(nn.Module):
    """
    Each of the 4 truncation levels is treated as one token.
    Token_i = 8 statistical features from tau_i% of the cycle.
    A learnable positional embedding encodes the completion level.
    Self-attention learns which completion stage carries the most
    reliable degradation signal — this is the core novelty.
    """
    def __init__(self, n_levels=4, feat_per_level=8, d_model=64,
                 nhead=4, num_layers=2, dropout=0.1):
        super().__init__()
        self.n_levels       = n_levels
        self.feat_per_level = feat_per_level

        # Project each token from feat_per_level → d_model
        self.input_proj = nn.Linear(feat_per_level, d_model)

        # Learnable positional embedding (one per truncation level)
        self.pos_emb = nn.Embedding(n_levels, d_model)

        # Transformer encoder
        enc_layer   = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # Cycle index processed separately and fused
        self.cycle_proj = nn.Linear(1, d_model)

        # Regression head
        self.head = nn.Sequential(
            nn.Linear(d_model * (n_levels + 1), 128), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x: [B, 33] — 32 truncation features + 1 cycle index
        x_trunc = x[:, :32].view(-1, self.n_levels, self.feat_per_level)  # [B, 4, 8]
        x_cycle = x[:, 32:33]                                              # [B, 1]

        # Project tokens + add positional embeddings
        pos   = torch.arange(self.n_levels, device=x.device).unsqueeze(0)  # [1, 4]
        tok   = self.input_proj(x_trunc) + self.pos_emb(pos)               # [B, 4, d_model]

        # Self-attention over truncation levels
        tok   = self.transformer(tok)   # [B, 4, d_model]

        # Cycle index embedding
        cyc   = self.cycle_proj(x_cycle).unsqueeze(1)  # [B, 1, d_model]

        # Concatenate all tokens + cycle context
        combined = torch.cat([tok, cyc], dim=1)         # [B, 5, d_model]
        flat     = combined.flatten(1)                  # [B, 5*d_model]

        return self.head(flat).squeeze(-1)              # [B]

print('TAT parameter count:', sum(p.numel() for p in TruncationAwareTransformer().parameters()))

In [ ]:
def monotonicity_loss(y_pred, bat_ids, cyc_nums):
    """Penalize SoH increases between consecutive cycles within the same battery."""
    total   = y_pred.sum() * 0.0   # zero that stays in the graph (not a leaf)
    n_pairs = 0
    for bat in np.unique(bat_ids):
        mask  = bat_ids == bat
        if mask.sum() < 2:
            continue
        order = np.argsort(cyc_nums[mask])
        idx   = np.where(mask)[0][order]
        p     = y_pred[idx]
        diffs = p[1:] - p[:-1]          # should be <= 0 (decreasing SoH)
        viol  = torch.clamp(diffs, min=0.0)
        total   = total + (viol ** 2).sum()
        n_pairs += len(diffs)
    return total / max(n_pairs, 1)


def train_tat(model, X_tr, y_tr, bat_tr, cyc_tr, X_vl, y_vl,
              epochs=250, lr=1e-3, batch=32, lam=0.05):
    model  = model.to(device)
    opt    = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched  = optim.lr_scheduler.StepLR(opt, step_size=60, gamma=0.5)
    Xv     = torch.from_numpy(X_vl).to(device)
    yv     = torch.from_numpy(y_vl).to(device)

    N  = len(y_tr)
    mse_log, phy_log, val_log = [], [], []

    for epoch in range(epochs):
        model.train()
        perm = np.random.permutation(N)
        ep_mse = ep_phy = 0
        n_batches = 0

        for start in range(0, N - batch, batch):
            bi   = perm[start:start+batch]
            xb   = torch.from_numpy(X_tr[bi]).to(device)
            yb   = torch.from_numpy(y_tr[bi]).to(device)

            opt.zero_grad()
            pred = model(xb)

            mse  = nn.MSELoss()(pred, yb)
            phy  = monotonicity_loss(pred, bat_tr[bi], cyc_tr[bi])
            loss = mse + lam * phy
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            ep_mse += mse.item(); ep_phy += phy.item(); n_batches += 1

        sched.step()
        mse_log.append(ep_mse / n_batches)
        phy_log.append(ep_phy / n_batches)
        model.eval()
        with torch.no_grad():
            val_log.append(nn.MSELoss()(model(Xv), yv).item())

        if (epoch+1) % 50 == 0:
            print(f'Epoch {epoch+1}/{epochs} — MSE: {mse_log[-1]:.5f} | '
                  f'Physics: {phy_log[-1]:.5f} | Val: {val_log[-1]:.5f}')

    return mse_log, phy_log, val_log

torch.manual_seed(42)
tat = TruncationAwareTransformer()
mse_log, phy_log, val_log = train_tat(
    tat, X_tr, y_tr, bat_tr, cyc_tr, X_vl, y_vl, lam=0.05
)

tat.eval()
with torch.no_grad():
    y_pred_tat = tat(torch.from_numpy(X_te).to(device)).cpu().numpy()
rmse_tat, mae_tat, mape_tat = calc_metrics(y_te, y_pred_tat)
results['TAT (ours)'] = dict(rmse=rmse_tat, mae=mae_tat, mape=mape_tat, pred=y_pred_tat)
print(f'\nTAT — RMSE: {rmse_tat:.4f} | MAE: {mae_tat:.4f} | MAPE: {mape_tat:.2f}%')

## 9. Ablation Study

In [ ]:
ablation_results = {}

# Variant 1: TAT, full cycle only (last 8 features + cycle idx)
X_tr_fc = np.hstack([X_tr[:, 24:32], X_tr[:, 32:33]]).astype(np.float32)
X_vl_fc = np.hstack([X_vl[:, 24:32], X_vl[:, 32:33]]).astype(np.float32)
X_te_fc = np.hstack([X_te[:, 24:32], X_te[:, 32:33]]).astype(np.float32)

class TATFullCycleOnly(nn.Module):
    """TAT variant: single truncation level (full cycle only), with physics loss."""
    def __init__(self, input_dim=9):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 32),         nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

torch.manual_seed(42)
m_fc = TATFullCycleOnly(9).to(device)
train_tat(m_fc, X_tr_fc, y_tr, bat_tr, cyc_tr, X_vl_fc, y_vl, lam=0.05, epochs=250)
m_fc.eval()
with torch.no_grad():
    p = m_fc(torch.from_numpy(X_te_fc).to(device)).cpu().numpy()
r, m, _ = calc_metrics(y_te, p)
ablation_results['Full cycle only + physics'] = dict(rmse=r, mae=m)
print(f'Full cycle only + physics: RMSE={r:.4f} MAE={m:.4f}')

# Variant 2: TAT, multi-truncation, no physics
torch.manual_seed(42)
tat_nop = TruncationAwareTransformer().to(device)
train_tat(tat_nop, X_tr, y_tr, bat_tr, cyc_tr, X_vl, y_vl, lam=0.0, epochs=250)
tat_nop.eval()
with torch.no_grad():
    p = tat_nop(torch.from_numpy(X_te).to(device)).cpu().numpy()
r, m, _ = calc_metrics(y_te, p)
ablation_results['Multi-truncation, no physics'] = dict(rmse=r, mae=m)
print(f'Multi-truncation, no physics: RMSE={r:.4f} MAE={m:.4f}')

# Variant 3: MLP (full pipeline, no physics, no transformer)
ablation_results['MLP (no truncation structure, no physics)'] = dict(
    rmse=rmse_mlp, mae=mae_mlp
)

print('\nAblation summary:')
for k, v in ablation_results.items():
    print(f'  {k}: RMSE={v["rmse"]:.4f} MAE={v["mae"]:.4f}')

## 10. Per-Truncation Level Evaluation

In [ ]:
trunc_labels = ['30%', '50%', '75%', '100%']
rmse_trunc, mae_trunc = [], []

tat.eval()
for t_idx in range(4):
    X_masked = X_te.copy()
    # Zero out all truncation features except the current level
    for other in range(4):
        if other != t_idx:
            X_masked[:, other*8:(other+1)*8] = 0.0
    with torch.no_grad():
        p = tat(torch.from_numpy(X_masked).to(device)).cpu().numpy()
    r, m, _ = calc_metrics(y_te, p)
    rmse_trunc.append(r); mae_trunc.append(m)
    print(f'{trunc_labels[t_idx]} truncation — RMSE: {r:.4f} | MAE: {m:.4f}')

# Full model
with torch.no_grad():
    p_full = tat(torch.from_numpy(X_te).to(device)).cpu().numpy()
r_full, m_full, _ = calc_metrics(y_te, p_full)
print(f'Full (all levels) — RMSE: {r_full:.4f} | MAE: {m_full:.4f}')

## 11. Multi-Seed Variance

In [ ]:
N_SEEDS = 5
rmse_seeds, mae_seeds = [], []

for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)
    m = TruncationAwareTransformer().to(device)
    train_tat(m, X_tr, y_tr, bat_tr, cyc_tr, X_vl, y_vl, lam=0.05, epochs=250)
    m.eval()
    with torch.no_grad():
        p = m(torch.from_numpy(X_te).to(device)).cpu().numpy()
    r, mv, _ = calc_metrics(y_te, p)
    rmse_seeds.append(r); mae_seeds.append(mv)
    print(f'Seed {seed}: RMSE={r:.4f} MAE={mv:.4f}')

print(f'\nRMSE: {np.mean(rmse_seeds):.4f} ± {np.std(rmse_seeds):.4f}')
print(f'MAE:  {np.mean(mae_seeds):.4f} ± {np.std(mae_seeds):.4f}')

## 12. Lambda Sensitivity

In [ ]:
lambdas = [0.0, 0.01, 0.05, 0.1, 0.3, 0.5]
rmse_lam, mae_lam = [], []

for lam in lambdas:
    torch.manual_seed(1)
    m = TruncationAwareTransformer().to(device)
    train_tat(m, X_tr, y_tr, bat_tr, cyc_tr, X_vl, y_vl, lam=lam, epochs=250)
    m.eval()
    with torch.no_grad():
        p = m(torch.from_numpy(X_te).to(device)).cpu().numpy()
    r, mv, _ = calc_metrics(y_te, p)
    rmse_lam.append(r); mae_lam.append(mv)
    print(f'lambda={lam:.3f}: RMSE={r:.4f} MAE={mv:.4f}')

## 13. Paper Figures (all saved inline)

In [ ]:
os.makedirs('/kaggle/working/figures', exist_ok=True)
COLORS = {'Linear Regression':'royalblue','MLP':'tomato',
          'LSTM':'darkorange','TAT (ours)':'seagreen'}

# ── Fig 1: SoH prediction comparison ─────────────────────────────────────────
n_plot = min(300, len(y_te))
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(y_te[:n_plot], 'k-', lw=1.8, label='Actual SoH', zorder=5)
for name, res in results.items():
    ls = '--' if name != 'TAT (ours)' else '-'
    ax.plot(res['pred'][:n_plot], ls, lw=1.2, color=COLORS[name], label=name)
ax.set_xlabel('Test Sample Index'); ax.set_ylabel('State of Health')
ax.set_title('Battery SoH Estimation: Model Comparison')
ax.legend(loc='best', fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/fig_soh_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Fig 2: Metrics bar chart ──────────────────────────────────────────────────
model_names = list(results.keys())
rmse_vals   = [results[k]['rmse'] for k in model_names]
mae_vals    = [results[k]['mae']  for k in model_names]
x = np.arange(len(model_names))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 3.5))
bars1 = ax.bar(x - w/2, rmse_vals, w, label='RMSE', color='steelblue')
bars2 = ax.bar(x + w/2, mae_vals,  w, label='MAE',  color='coral')
ax.set_xticks(x); ax.set_xticklabels(model_names, fontsize=9)
ax.set_ylabel('Error'); ax.set_title('Performance Comparison')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
for b in list(bars1)+list(bars2):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.002,
            f'{b.get_height():.4f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/fig_metrics.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Fig 3: Scatter actual vs predicted (TAT) ─────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_te, y_pred_tat, s=10, alpha=0.4, color='seagreen')
mn, mx = min(y_te.min(), y_pred_tat.min()), max(y_te.max(), y_pred_tat.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=1.5)
ax.set_xlabel('Actual SoH'); ax.set_ylabel('Predicted SoH')
ax.set_title('TAT: Actual vs. Predicted SoH')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/fig_scatter.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Fig 4: Training loss curves ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(mse_log, 'b-', lw=1.5, label='MSE Loss')
ax.plot(phy_log, 'r--', lw=1.5, label='Physics (Monotonicity) Loss')
ax.plot(val_log, 'g:', lw=1.5, label='Validation Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('TAT Training Loss Curves')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/fig_loss.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Fig 5: Per-truncation RMSE ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(trunc_labels + ['All levels'], rmse_trunc + [r_full],
       color=['#aec6cf']*4 + ['seagreen'])
ax.set_xlabel('Cycle Completion Level'); ax.set_ylabel('RMSE')
ax.set_title('RMSE vs. Cycle Completion Level (TAT)')
ax.grid(True, axis='y', alpha=0.3)
for i, v in enumerate(rmse_trunc + [r_full]):
    ax.text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/fig_truncation_rmse.png', dpi=200, bbox_inches='tight')
plt.show()

# ── Fig 6: Lambda sensitivity ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(lambdas, rmse_lam, 'bo-', lw=1.5, ms=6, label='RMSE')
ax.plot(lambdas, mae_lam,  'rs-', lw=1.5, ms=6, label='MAE')
ax.axvline(0.05, color='gray', linestyle='--', lw=1, label='Selected λ=0.05')
ax.set_xlabel('λ (physics loss weight)'); ax.set_ylabel('Error')
ax.set_title('Sensitivity to Physics Loss Weight λ')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/figures/fig_lambda.png', dpi=200, bbox_inches='tight')
plt.show()

print('All figures saved to /kaggle/working/figures/')

## 14. Results Summary Tables

In [ ]:
print('='*60)
print('TABLE I: Main Results')
print(f'{"Model":<22} {"RMSE":>8} {"MAE":>8} {"MAPE(%)":>10}')
print('-'*50)
for name, res in results.items():
    print(f'{name:<22} {res["rmse"]:>8.4f} {res["mae"]:>8.4f} {res["mape"]:>10.2f}')

print(f'\nImprovement over MLP:')
print(f'  RMSE: {(rmse_mlp-rmse_tat)/rmse_mlp*100:.1f}%')
print(f'  MAE:  {(mae_mlp-mae_tat)/mae_mlp*100:.1f}%')

print('\n' + '='*60)
print('TABLE II: Ablation Study')
print(f'{"Variant":<40} {"RMSE":>8} {"MAE":>8}')
print('-'*58)
for name, res in ablation_results.items():
    print(f'{name:<40} {res["rmse"]:>8.4f} {res["mae"]:>8.4f}')
print(f'{"TAT (full, ours)":<40} {rmse_tat:>8.4f} {mae_tat:>8.4f}')

print('\n' + '='*60)
print('TABLE III: Per-Truncation Level (TAT)')
print(f'{"Truncation Level":<20} {"RMSE":>8} {"MAE":>8}')
print('-'*38)
for i, lbl in enumerate(trunc_labels):
    print(f'{lbl:<20} {rmse_trunc[i]:>8.4f} {mae_trunc[i]:>8.4f}')
print(f'{"All levels":<20} {r_full:>8.4f} {m_full:>8.4f}')

print('\n' + '='*60)
print('Multi-seed (n=5):')
print(f'  RMSE: {np.mean(rmse_seeds):.4f} ± {np.std(rmse_seeds):.4f}')
print(f'  MAE:  {np.mean(mae_seeds):.4f} ± {np.std(mae_seeds):.4f}')